<a href="https://colab.research.google.com/github/jaqchien-dotcom/Jacob1-TW/blob/main/02_%E5%A4%96%E9%83%A8%E6%95%B8%E6%93%9A%E5%B0%8D%E6%8E%A5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 外部數據對接

黃彬華編撰

## RESTful 概論
RESTful 是一種設計 Web API 的風格，源自於 Roy Fielding 在博士論文中提出的 REST (Representational State Transfer) 概念

RESTful 設計風格

URL 表示資源 (resource)
* 例如書是資源：/books
* 資源的資料格式：通常是 JSON

HTTP 方法表示操作
* GET：查詢、取得資料
* POST：新增資料
* PUT：修改資料
* DELETE：刪除資料

RESTful 設計風格範例如下表所示：

| 功能 | HTTP 方法 (動作) | URL(資源)|
| --- | --- | --- |
| 取得所有書 | GET | /books |
| 查詢一本書 | GET | /books/{isbn}
| 新增書籍 | POST | /books
| 修改書籍 | PUT | /books/{isbn}
| 刪除書籍 | DELETE | /books/{isbn}


### RESTful Server Demo

下方 server 程式展示 RESTful 設計風格

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
from threading import Thread
import uvicorn
import nest_asyncio

# Colab 系統本身已經有正在執行的 event loop。
# 而 uvicorn(FastAPI server) 啟動時也會想建立自己的 event loop，
# 若不處理，可能出現：RuntimeError: This event loop is already running。
# nest_asyncio.apply() 會修改 asyncio 行為，允許巢狀 (nested) event loop 運作
nest_asyncio.apply()

app = FastAPI()


# 配合 FastAPI，必須使用 BaseModel
class Book(BaseModel):
    isbn: str
    name: str
    price: int
    author: str


# list 儲存測試資料
books: List[Book] = [
    Book(isbn="111", name="Python", price=500, author="Paul"),
    Book(isbn="222", name="Java", price=500, author="John")
]


# 取得所有書
@app.get("/books", response_model=List[Book])
def get_books():
    return books


# 取得單一本書
@app.get("/books/{isbn}", response_model=Book)
def get_book(isbn: str):
    for book in books:
        if book.isbn == isbn:
            return book
    raise HTTPException(status_code=404, detail="Book not found")


# 新增書
@app.post("/books", response_model=Book)
def create_book(book: Book):
    # 檢查重複 ISBN
    for b in books:
        if b.isbn == book.isbn:
            raise HTTPException(status_code=400, detail="ISBN already exists")
    books.append(book)
    return book


# 更新書
@app.put("/books/{isbn}", response_model=Book)
def update_book(isbn: str, updated_book: Book):
    # enumerate() 會同時給予 index 與 object
    for i, book in enumerate(books):
        if book.isbn == isbn:
            books[i] = updated_book
            return updated_book
    raise HTTPException(status_code=404, detail="Book not found")


# 刪除書
@app.delete("/books/{isbn}")
def delete_book(isbn: str):
    for i, book in enumerate(books):
        if book.isbn == isbn:
            del books[i]
            return {"message": "Book deleted"}
    raise HTTPException(status_code=404, detail="Book not found")


# 定義一個函式 run_server()
# 目的是把 FastAPI server 啟動程式包裝起來，
# 之後交給背景執行緒 (Thread) 執行。
def run_server():
    # 「host="0.0.0.0"」允許各種網路連線，方便測試；「host="127.0.0.1"」只允許本機連線
    uvicorn.run(app, host="0.0.0.0", port=8000)

# 使用 Uvicorn 啟動 FastAPI server
Thread(target=run_server).start()
print("FastAPI 伺服器已於 http://0.0.0.0:8000 背景啟動")

FastAPI 伺服器已於 http://0.0.0.0:8000 背景啟動


### RESTful Client Demo
注意：
如果列印出：「INFO:     127.0.0.1:55382 - "GET /books/111 HTTP/1.1" 200 OK」
並不是目前的這支 Python client 程式列印出來的，而是背後運行的伺服器列印出來的存取日誌（Access Log）。這通常發生在同一個終端機（Terminal）或同一個 Colab 儲存格中，同時運行了 server 與這支 client 程式。

In [ ]:
# 匯入 requests 套件，用來發送 HTTP request
import requests

# RESTful API server URL
BASE_URL = "http://127.0.0.1:8000"


# Book 類別
class Book:
    def __init__(self, isbn, name, price, author):
        self.isbn = isbn
        self.name = name
        self.price = price
        self.author = author

    def __repr__(self):
        return f"{self.isbn}, {self.name}, {self.price}, {self.author}"


# 取得所有書籍並顯示
def list_books():
    # 發送 GET request
    response = requests.get(f"{BASE_URL}/books")

    # HTTP 狀態碼非 2xx 時產生例外
    response.raise_for_status()

    # JSON -> Python list
    books = response.json()

    print("\n=== Book List ===")

    # 每筆 dict 轉成 Book 物件
    for b in books:
        # 使用 dict unpacking
        book = Book(**b)
        print(book)


# 查詢單一本書
def get_book():
    isbn = input("ISBN: ")

    # 發送 GET request
    response = requests.get(f"{BASE_URL}/books/{isbn}")

    # HTTP 狀態碼非 2xx 時產生例外
    response.raise_for_status()

    # JSON -> dict
    data = response.json()

    return data


# 新增書籍
def add_book():
    # 建立 Book 物件
    book = Book(
        input("ISBN: "),
        input("Name: "),
        int(input("Price: ")),
        input("Author: ")
    )

    # 發送 POST request
    response = requests.post(
        f"{BASE_URL}/books",
        # Book -> dict -> JSON
        json=vars(book)
    )

    # HTTP 狀態碼非 2xx 時產生例外
    response.raise_for_status()

    return response.json()


# 更新書籍
def update_book():
    isbn = input("ISBN to update: ")

    # 建立更新後的 Book 物件
    book = Book(
        isbn,
        input("New Name: "),
        int(input("New Price: ")),
        input("New Author: ")
    )

    # 發送 PUT request
    response = requests.put(
        f"{BASE_URL}/books/{isbn}",
        json=vars(book)
    )

    # HTTP 狀態碼非 2xx 時產生例外
    response.raise_for_status()

    return response.json()


# 刪除書籍
def delete_book():
    isbn = input("ISBN to delete: ")

    # 發送 DELETE request
    response = requests.delete(f"{BASE_URL}/books/{isbn}")

    # HTTP 狀態碼非 2xx 時產生例外
    response.raise_for_status()

    return response.json()


# 主選單
def main():
    while True:
        try:
            print("\n=== Book System ===")
            print("1. List all books")
            print("2. Get book")
            print("3. Add book")
            print("4. Update book")
            print("5. Delete book")
            print("0. Exit")

            choice = input("Choose: ")

            match choice:

                case "1":
                    # 取得所有書籍並顯示
                    list_books()

                case "2":
                    # 查詢單一本書
                    data = get_book()
                    book = Book(**data)
                    print(book)

                case "3":
                    # 新增書籍
                    result = add_book()
                    print(result)

                case "4":
                    # 更新書籍
                    result = update_book()
                    print(result)

                case "5":
                    # 刪除書籍
                    result = delete_book()
                    print(result)

                case "0":
                    break

                case _:
                    print("Invalid choice")

        # HTTP 狀態碼錯誤（404 / 500 等）
        except requests.HTTPError as e:
            print(
                f"HTTP {e.response.status_code}：",
                e.response.json().get("detail", e.response.text)
            )

        # 網路錯誤（斷線 / timeout / DNS）
        except requests.RequestException as e:
            print("網路錯誤：", e)

        # 型別錯誤（例如 int(input()) 失敗）
        except ValueError:
            print("輸入錯誤：Price 必須為數字")


# 程式進入點
main()

INFO:     Started server process [1037]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



=== Book System ===
1. List all books
2. Get book
3. Add book
4. Update book
5. Delete book
0. Exit
Choose: 1
INFO:     127.0.0.1:54632 - "GET /books HTTP/1.1" 200 OK

=== Book List ===
111, Python, 500, Paul
222, Java, 500, John

=== Book System ===
1. List all books
2. Get book
3. Add book
4. Update book
5. Delete book
0. Exit


## Payload 封裝

**先了解什麼是 Payload**

Payload 是指在資料傳輸過程中，真正要傳送的資料內容，不包含協定本身所需的控制資訊。

例如在網路通訊中，一個封包（Packet）通常可分成：
* Header（標頭）：控制資訊
* Payload（資料負載）：實際資料

示意圖：
```
HTTP Header
├─ Content-Type: application/json
├─ Authorization: Bearer xxx
└─ ...

Payload
{
    "username": "ken",
    "password": "1234"
}
```

**Payload 封裝（Payload Encapsulation）**

就是將資料包裝成特定格式，以便傳輸或儲存。

例如使用者登入：
```
username = "ken"
password = "1234"
```
Python 可將資料封裝成 JSON Payload：
```
payload = {
    "username": username,
    "password": password
}
```
然後送出：
```
requests.post(
    url,
    json=payload
)
```
此時的 payload 就是封裝後的資料。

### Payload Server Demo

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
from threading import Thread
import uvicorn
import nest_asyncio

nest_asyncio.apply()

app_payload = FastAPI()

# 商品類別，配合 FastAPI，必須使用 BaseModel
class Product(BaseModel):
    id: int
    name: str
    description: str | None = None
    price: int
    tax: float | None = None

# 模擬商品資料庫
products_db = [
    Product(id=1, name="筆電", description="功能強大筆電", price=36000, tax=0.08),
    Product(id=2, name="滑鼠", description="無線滑鼠", price=500)
]


# 回傳所有商品
# 「response_model=List[Product]」代表輸出的 Payload 結構必須是 List[Product]
# FastAPI 最後會將 List[Product] 資料自動封裝成 JSON 然後成為 HTTP Response 的 Payload
@app_payload.get("/products/", response_model=List[Product])
async def get_products():
    # 執行到 return 這行，FastAPI 會檢查 products_db 是否符合 List[Product] 類型；
    # 如果符合最終序列化為 JSON Payload 傳送給前端
    return products_db

# 新增商品
@app_payload.post("/products/", response_model=Product)
# 接收到前端傳來的 JSON Payload 並反序列化 (還原) 為 Product 物件
# 轉換時，它會自動檢查格式。如果前端傳入的 price 不是數字，API 會直接報錯（422 Unprocessable Entity）
async def create_product(product: Product):
    products_db.append(product)
    # FastAPI 根據「response_model=Product」，將這個 product 物件序列化成 JSON Payload 傳送給前端
    return product



# 啟動 FastAPI 伺服器 (使用不同的端口以避免衝突)
def run_payload_server():
    uvicorn.run(app_payload, host="0.0.0.0", port=8001)

print("正在啟動 Payload Demo FastAPI 伺服器於 http://0.0.0.0:8001")
Thread(target=run_payload_server).start()
print("Payload Demo 伺服器已在背景啟動。")
print("您可以向 /products/ 發送 POST 請求來新增商品，或向 /products/ 發送 GET 請求來獲取所有商品。")

正在啟動 Payload Demo FastAPI 伺服器於 http://0.0.0.0:8001
Payload Demo 伺服器已在背景啟動。
您可以向 /products/ 發送 POST 請求來新增商品，或向 /products/ 發送 GET 請求來獲取所有商品。


### Payload Client Demo

下方 client 程式會與上方 server 程式互動。將展示如何發送包含封裝 Payload 的 POST 請求，以及如何接收和處理封裝的 JSON 回應。

In [ ]:
import requests
import json

PAYLOAD_BASE_URL = "http://127.0.0.1:8001"

# 測試 GET /products/
print("\n--- 測試 GET /products/ (獲取所有商品) ---")
try:
    response = requests.get(f"{PAYLOAD_BASE_URL}/products/")
    # 檢查是否有 HTTP 錯誤
    response.raise_for_status()
    products = response.json()
    print("獲取到的商品列表：")
    for product in products:
        print(json.dumps(product, ensure_ascii=False, indent=2))
except requests.exceptions.RequestException as e:
    print(f"GET 請求失敗: {e}")

# 測試 POST /products/ (新增商品)
print("\n--- 測試 POST /products/ (新增商品) ---")
new_product_data = {
    "id": 3,
    "name": "鍵盤",
    "description": "優質無線鍵盤",
    "price": 450,
    "tax": 0.05
}

try:
    response = requests.post(f"{PAYLOAD_BASE_URL}/products/", json=new_product_data)
    response.raise_for_status()
    created_product = response.json()
    print("成功新增商品：")
    print(json.dumps(created_product, ensure_ascii=False, indent=2))

    # 再次獲取所有商品，確認新商品已添加
    print("\n--- 再次獲取所有商品以確認新商品 ---")
    response = requests.get(f"{PAYLOAD_BASE_URL}/products/")
    response.raise_for_status()
    products_after_add = response.json()
    print("新增後的商品列表：")
    for product in products_after_add:
        print(json.dumps(product, ensure_ascii=False, indent=2))

except requests.exceptions.RequestException as e:
    print(f"POST 請求失敗: {e}")

# 測試 POST /products/ (帶有無效數據，觀察資料驗證情形)
print("\n--- 測試 POST /products/ (帶有無效數據) ---")
invalid_product_data = {
    "id": "invalid_id", # id 需為 int
    "name": "22吋螢幕",
    "price": "not_a_number" # price 需為 int
}

try:
    response = requests.post(f"{PAYLOAD_BASE_URL}/products/", json=invalid_product_data)
    response.raise_for_status() # 這將會引發 HTTPError，因為狀態碼不是 2xx
    print("意外：成功新增商品，但數據無效！")
except requests.exceptions.HTTPError as e:
    print(f"預期錯誤：POST 請求失敗，狀態碼 {e.response.status_code}")
    print(f"錯誤詳情：{e.response.json()}")
except requests.exceptions.RequestException as e:
    print(f"POST 請求失敗: {e}")


--- 測試 GET /products/ (獲取所有商品) ---
INFO:     127.0.0.1:57808 - "GET /products/ HTTP/1.1" 200 OK
獲取到的商品列表：
{
  "id": 1,
  "name": "筆電",
  "description": "功能強大筆電",
  "price": 36000,
  "tax": 0.08
}
{
  "id": 2,
  "name": "滑鼠",
  "description": "無線滑鼠",
  "price": 500,
  "tax": null
}

--- 測試 POST /products/ (新增商品) ---
INFO:     127.0.0.1:57820 - "POST /products/ HTTP/1.1" 200 OK
成功新增商品：
{
  "id": 3,
  "name": "鍵盤",
  "description": "優質無線鍵盤",
  "price": 450,
  "tax": 0.05
}

--- 再次獲取所有商品以確認新商品 ---
INFO:     127.0.0.1:57828 - "GET /products/ HTTP/1.1" 200 OK
新增後的商品列表：
{
  "id": 1,
  "name": "筆電",
  "description": "功能強大筆電",
  "price": 36000,
  "tax": 0.08
}
{
  "id": 2,
  "name": "滑鼠",
  "description": "無線滑鼠",
  "price": 500,
  "tax": null
}
{
  "id": 3,
  "name": "鍵盤",
  "description": "優質無線鍵盤",
  "price": 450,
  "tax": 0.05
}

--- 測試 POST /products/ (帶有無效數據) ---
INFO:     127.0.0.1:57840 - "POST /products/ HTTP/1.1" 422 Unprocessable Entity
預期錯誤：POST 請求失敗，狀態碼 422
錯誤詳情：{'detail': [{'type':

## 複雜 JSON 嵌套解析

JSON (JavaScript Object Notation) 是一種輕量級的數據交換格式，讓人易於閱讀和編寫，也易於機器解析和生成。JSON 基於兩種結構：

* Array：有序性資料，相當於 Python 的 list
* Object：key-value 型式，相當於 Python 的 dictionary

JSON 嵌套解析是指從多層巢狀 (nested) 的 JSON 結構中，逐層取出需要的資料。

在實際的 Web 開發和數據交換中，為了表示真實世界中複雜的實體關係，JSON 數據經常會以嵌套的形式出現。例如，一個訂單 (Order) 可能包含多個訂單項目 (Order Items)，每個訂單項目又包含一個商品 (Product) 及其詳細訊息。

In [ ]:
# 模擬 server 回傳資料
response = {
    "status": "success",
    "data": {
        "order": {
            "order_id": "ORD20260617001",
            "order_date": "2026-06-17",
            "customer": {
                "id": 1,
                "name": "Ken"
            },
            "items": [
                {
                    "isbn": "978-0001",
                    "book_name": "Python 入門",
                    "price": 500,
                    "quantity": 2
                },
                {
                    "isbn": "978-0002",
                    "book_name": "SQL 精通",
                    "price": 600,
                    "quantity": 1
                }
            ]
        }
    }
}

print("訂單編號:", response["data"]["order"]["order_id"])
print("訂單日期:", response["data"]["order"]["order_date"])
print("訂購者名稱:", response["data"]["order"]["customer"]["name"])
print("第一本書的書名:", response["data"]["order"]["items"][0]["book_name"])
print("第一本書的價格:", response["data"]["order"]["items"][0]["price"])
print("第一本書訂購數量:", response["data"]["order"]["items"][0]["quantity"])

訂單編號: ORD20260617001
訂單日期: 2026-06-17
訂購者名稱: Ken
第一本書的書名: Python 入門
第一本書的價格: 500
第一本書訂購數量: 2



複雜 JSON 嵌套解析 (Complex JSON Nested Parsing) 就是指從這種多層次的 JSON 數據中提取特定資訊的過程。這通常需要：

*   逐層遍歷：根據數據的結構，一步步地進入到所需的數據層次。
*   透過 key 取值：對於 JSON Object，搭配 key 取值。
*   透過 index 取值：對於 JSON Array，搭配迴圈取值。
*   錯誤處理：訪問不存在的 key 時，需要有適當的錯誤處理機制，以避免程序崩潰。例如使用 dict.get() 方法或 try-except 區塊來處理可能不存在的 key

In [ ]:
import requests


def fetch_users():
    # 測試 JSON 格式網址 (說明網址：https://jsonplaceholder.typicode.com/?utm_source=chatgpt.com)
    url = "https://jsonplaceholder.typicode.com/users"

    try:
        response = requests.get(url, timeout=30)
        # HTTP 狀態碼檢查
        response.raise_for_status()
        # 取得所有使用者，依照 JSON 內容會轉成 list
        users = response.json()
        return users

    except requests.exceptions.Timeout:
        print("連線逾時")
        return []

    except requests.exceptions.ConnectionError as e:
        print(f"無法連線至伺服器: {e}")
        return []

    except requests.exceptions.HTTPError as e:
        print(f"HTTP 錯誤：{e}")
        return []

    except Exception as e:
        print(f"發生未知錯誤：{e}")
        return []


def clean_value(value):
    """
    統一處理空值：
    None / 空字串 / 空白 → 無資料
    """

    if value is None:
        return "無資料"

    # value.strip() 如果變成 ""，布林值評估時會被視為 False
    if isinstance(value, str) and not value.strip():
        return "無資料"

    return value


# 只顯示前 5 位使用者
def print_first_5_users(users):
    print(f"使用者總數：{len(users)}")
    print("只顯示前 5 位")
    # 對於 JSON Array (相當於 list)，搭配迴圈取值
    # 「start=1」指定 index 初始值為 1，預設為 0
    for index, user in enumerate(users[:5], start=1):
        print("=" * 30)
        print(f"第 {index} 位使用者")
        # 對於 JSON Object (相當於 dict)，搭配 key 取值
        # 若無 key 回傳 None，不會發生例外 (Exception)
        name = clean_value(user.get("name"))
        username = clean_value(user.get("username"))
        email = clean_value(user.get("email"))
        phone = clean_value(user.get("phone"))
        website = clean_value(user.get("website"))

        # 若無 key 回傳指定預設值，不會發生例外 (Exception)
        address = user.get("address", {})
        city = clean_value(address.get("city"))
        street = clean_value(address.get("street"))
        suite = clean_value(address.get("suite"))

        company = user.get("company", {})
        company_name = clean_value(company.get("name"))

        print("姓名:", name)
        print("帳號:", username)
        print("Email:", email)
        print("電話:", phone)
        print("網站:", website)
        print("城市:", city)
        print("街道:", street)
        print("門牌/套房:", suite)
        print("公司:", company_name)


def main():
    users = fetch_users()

    if not users:
        print("沒有取得任何資料")
        return

    print_first_5_users(users)


main()

使用者總數：10
只顯示前 5 位
第 1 位使用者
姓名: Leanne Graham
帳號: Bret
Email: Sincere@april.biz
電話: 1-770-736-8031 x56442
網站: hildegard.org
城市: Gwenborough
街道: Kulas Light
門牌/套房: Apt. 556
公司: Romaguera-Crona
第 2 位使用者
姓名: Ervin Howell
帳號: Antonette
Email: Shanna@melissa.tv
電話: 010-692-6593 x09125
網站: anastasia.net
城市: Wisokyburgh
街道: Victor Plains
門牌/套房: Suite 879
公司: Deckow-Crist
第 3 位使用者
姓名: Clementine Bauch
帳號: Samantha
Email: Nathan@yesenia.net
電話: 1-463-123-4447
網站: ramiro.info
城市: McKenziehaven
街道: Douglas Extension
門牌/套房: Suite 847
公司: Romaguera-Jacobson
第 4 位使用者
姓名: Patricia Lebsack
帳號: Karianne
Email: Julianne.OConner@kory.org
電話: 493-170-9623 x156
網站: kale.biz
城市: South Elvis
街道: Hoeger Mall
門牌/套房: Apt. 692
公司: Robel-Corkery
第 5 位使用者
姓名: Chelsey Dietrich
帳號: Kamren
Email: Lucio_Hettinger@annie.ca
電話: (254)954-1289
網站: demarco.info
城市: Roscoeview
街道: Skiles Walks
門牌/套房: Suite 351
公司: Keebler LLC


In [ ]:
from pathlib import Path
import requests
import zipfile
import json


def fetch_restaurants():
    # 餐飲開放資料JSON格式網址 (說明網址：https://data.gov.tw/dataset/7779)
    url = "https://media.taiwan.net.tw/XMLReleaseAll_public/v2.0/Zh_tw/Restaurant-json.zip"

    download_dir = Path("download")
    download_dir.mkdir(exist_ok=True)

    zip_path = download_dir / "Restaurant-json.zip"
    extract_dir = download_dir / "restaurant_json"
    extract_dir.mkdir(exist_ok=True)

    try:
        # 下載 ZIP
        response = requests.get(url, timeout=30)

        # HTTP 狀態碼檢查
        response.raise_for_status()

        zip_path.write_bytes(response.content)
        print("下載完成:", zip_path)

        # 解壓縮 ZIP
        with zipfile.ZipFile(zip_path, "r") as zip_file:
            zip_file.extractall(extract_dir)

        print("解壓縮完成:", extract_dir)

        json_path = extract_dir / "RestaurantList.json"
        print("JSON 檔案路徑:", json_path)

        # 讀取 JSON
        with json_path.open("r", encoding="utf-8-sig") as f:
            data = json.load(f)

        # 取得所有餐廳資料
        restaurants = data["Restaurants"]

        return restaurants

    except requests.exceptions.Timeout:
        print("連線逾時")
        return []

    except requests.exceptions.ConnectionError:
        print("無法連線至伺服器")
        return []

    except requests.exceptions.HTTPError as e:
        print(f"HTTP 錯誤：{e}")
        return []

    except KeyError as e:
        print(f"JSON 結構異常：{e}")
        return []

    except Exception as e:
        print(f"發生未知錯誤：{e}")
        return []


def clean_value(value):
    """
    統一處理空值：
    None / 空字串 / 空白 → 無資料
    """

    if value is None:
        return "無資料"

    # value.strip() 如果變成 ""，布林值評估時會被視為 False
    if isinstance(value, str) and not value.strip():
        return "無資料"

    return value

# 只顯示前 5 家餐廳的重要資訊
def print_first_5_restaurants(restaurants):
    print(f"餐廳總數：{len(restaurants)}")
    print("只顯示前 5 家")
    for index, restaurant in enumerate(restaurants[:5], start=1):
        print("=" * 30)
        print(f"第 {index} 家餐廳")

        name = restaurant.get("RestaurantName")

        telephones = restaurant.get("Telephones", [])
        telephone = "無資料"
        if telephones:
            telephone = clean_value(telephones[0].get("Tel"))

        description = clean_value(restaurant.get("Description"))

        postal_address = restaurant.get("PostalAddress", {})
        city = clean_value(postal_address.get("City"))
        town = clean_value(postal_address.get("Town"))
        street = clean_value(postal_address.get("StreetAddress"))
        address = city + town + street

        website = clean_value(restaurant.get("WebsiteURL"))

        print("餐廳名:", name)
        print("電話:", telephone)
        print("描述:", description)
        print("地址:", address)
        print("網站:", website)


def main():
    restaurants = fetch_restaurants()

    if not restaurants:
        print("沒有取得任何資料")
        return

    print_first_5_restaurants(restaurants)


main()

下載完成: download/Restaurant-json.zip
解壓縮完成: download/restaurant_json
JSON 檔案路徑: download/restaurant_json/RestaurantList.json
餐廳總數：3690
只顯示前 5 家
第 1 家餐廳
餐廳名: 佳樂蛋糕
電話: (03)3335339
描述: 以波士頓派聞名美食界，並且讓網友大力推薦的佳樂蛋糕，鬆軟、甜而不膩的幸福口感，一直是桃園人的最愛，不論當下午茶點心還是生日蛋糕，都讓人面子裡子十足。建議遊客必點原味或巧克力口味的波士頓派，絕對令人意猶未盡。
地址: 桃園市桃園區民生路124號
網站: 無資料
第 2 家餐廳
餐廳名: 正記中崎本舖
電話: (03)3370880
描述: 第一屆桃園十大伴手禮-布丁蛋糕禮盒第二屆、第三屆桃園十大伴手禮-中崎布丁蛋糕逛完景福宮大廟，遊客們絕不會錯過廟後正記中崎本舖的中崎布丁蛋糕。道地傳統奶油式的蛋糕作法，細嫩香軟的精湛口味，是當地人生日蛋糕與下午茶的最佳選擇之一。馳名全台布丁蛋糕，歷年來榮獲無數食品評競金牌獎。口味獨特、用料實在、奶香濃郁、綿密細緻，獨一無二的口味廣受大眾喜愛，遠近馳名。
地址: 桃園市桃園區中正路219號
網站: 無資料
第 3 家餐廳
餐廳名: 郭祿食品有限公司
電話: (03)3385156
描述: 第一屆桃園十大伴手禮：郭祿月靜禮盒桃園老字號的伴手禮名店，以香Q又不黏牙的牛軋糖聞名全國，搭配香酥綿密口感的鳳梨酥與蛋黃酥，三項特色名點組成的月靜禮盒，不但吃出老師傅珍藏手藝的美味，送禮更是體面。第二屆桃園十大伴手禮：郭祿台灣情牛軋糖禮盒牛軋糖不黏牙充滿咬勁的本土特選香脆花生，鳳梨酥嚴選上等原料加上手工製造皮酥餡軟，甜而不膩，芝麻本身就有很多對身體有益的元素，選用上等芝麻加上特製麥芽，打開後芝麻香讓人想要一口接一口，軟硬適中。此外本禮盒獲得綠色環保認證，符合現代人環保觀念，送禮得體又大方。第三屆桃園十大伴手禮：溫馨禮盒第五屆桃園十大伴手禮：土地公禮盒
地址: 桃園市桃園區中山路252號
網站: 無資料
第 4 家餐廳
餐廳名: 郭記名點
電話: (03)3355135
描述: 榮獲桃園市十大伴手禮，以四十年的老到功力聞名，老闆郭先進夫妻和赴日鑽研的兒子郭雨函，用專業經營和精湛的烘培技術，將號稱名點三

## Rate Limit (頻率限制)

頻率限制 (Rate Limit) 是一種在 API 或服務中常見的機制，用於控制 client (client) 在一定時間內可以發送請求的次數。當 client 在超過允許的次數限制後繼續發送請求時，API 會拒絕這些請求，並通常返回一個錯誤代碼 (例如 HTTP 429 Too Many Requests)。

為什麼會有頻率限制？

* 防止濫用和攻擊：保護 API 免受惡意攻擊，如 DoS (拒絕服務) 攻擊或暴力破解。
* 資源保護：防止單一用戶過度消耗伺服器資源，確保所有用戶都能獲得穩定的服務。
* 成本控制：對於提供付費 API 的服務者來說，限制請求頻率有助於控制成本。


### Rate Limit Server Demo

下方 server 程式會模擬當 client 請求頻率過高時所發生限制的情境

In [ ]:
import requests
import time
import math
from fastapi import FastAPI, Request, Response, status
from threading import Thread
import uvicorn
import nest_asyncio


nest_asyncio.apply()

app_ratelimit = FastAPI()


# 限制時間
LIMIT_SECONDS = 10

# 每個 IP 在限制時間內最多請求次數
MAX_REQUESTS = 3

# 儲存各 Client 的資訊
#
# clients = {
#     "127.0.0.1": {
#         "count": 2,
#         "start_time": 1750212345.123
#     }
# }
#
clients = {}


@app_ratelimit.get("/limited-resource")
async def get_limited_resource(request: Request, response: Response):
    # 取得 client IP
    client_ip = request.client.host
    # 取得當下時間
    current_time = time.time()

    # 第一次出現的 IP
    if client_ip not in clients:
        clients[client_ip] = {
            "count": 0,
            "start_time": current_time
        }

    # 透過 IP 取出該 client 連線資訊
    client_info = clients[client_ip]

    # 計算 client 重置後的第一次請求已經過了多久時間
    elapsed = current_time - client_info["start_time"]

    # 超過時間限制則重置，讓請求次數歸零
    if elapsed >= LIMIT_SECONDS:
        client_info["count"] = 0
        client_info["start_time"] = current_time
        elapsed = 0

    # 累加請求次數
    client_info["count"] += 1
    current_count = client_info["count"]

    # 超過請求限制
    if current_count > MAX_REQUESTS:
        # 計算剩餘等待秒數，若有小數則進位
        retry_after = math.ceil(LIMIT_SECONDS - elapsed)

        # 不足一秒則視為一秒
        if retry_after < 1:
            retry_after = 1

        # 回應代碼設為 429
        response.status_code = status.HTTP_429_TOO_MANY_REQUESTS

        # 加上 "Retry-After" header
        response.headers[
            "Retry-After"
        ] = str(retry_after)

        return {
            "detail": "Too Many Requests",
            "ip": client_ip,
            "current_count": current_count,
            "retry_after_seconds": retry_after
        }


    # 正常回應 (未超過頻率限制)
    return {
        "message": "Success",
        "ip": client_ip,
        "current_count": current_count,
        "remaining_quota": MAX_REQUESTS - current_count
    }



# 啟動 FastAPI 伺服器 (使用不同的端口)
def run_ratelimit_server():
    uvicorn.run(app_ratelimit, host="0.0.0.0", port=8002)

print("正在啟動 Rate Limit Demo FastAPI 伺服器於 http://0.0.0.0:8002")
Thread(target=run_ratelimit_server).start()
print("Rate Limit Demo 伺服器已在背景啟動。")

正在啟動 Rate Limit Demo FastAPI 伺服器於 http://0.0.0.0:8002
Rate Limit Demo 伺服器已在背景啟動。


### Rate Limit Client Demo

如何應對頻率限制？

當遇到頻率限制時，client 不應該立即重試，而是應該遵循 API 提供的指示，或採用一些通用的策略：

* 檢查 `Retry-After` header (標頭)：許多 API 在返回 429 錯誤時，會在 HTTP 回應中包含一個 `Retry-After` header。這個 header 會告訴 client 應該等待多少秒才能再次發送請求。
* 指數退避 (Exponential Backoff)：這是一種常見的重試策略。當請求失敗並遇到頻率限制時，client 等待一段時間再重試。如果再次失敗，等待時間會以指數級增長。例如，第一次等待 1 秒，第二次等待 2 秒，第三次等待 4 秒，依此類推。同時，通常會設置一個最大等待時間和最大重試次數，以防止無限重試。
* 批量處理請求：如果可能，將多個小請求合併為一個大請求，可以減少請求總數。
* 暫存於 client：對於不常變化的數據，可以暫存於 client，減少不必要的 API 調用。

In [ ]:
import requests
import time
import random

BASE_URL = "http://127.0.0.1:8002"


def fetch_resource_with_retry(max_retries=3):
    retries = 0
    while retries <= max_retries:
        try:
            print(f"嘗試取得資源 (retry={retries})")
            response = requests.get(
                f"{BASE_URL}/limited-resource",
                timeout=5
            )

            # Rate Limit
            if response.status_code == 429:
                body = response.json()
                print("收到 429", body)

                # 取得 Retry-After 秒數
                retry_after = response.headers.get("Retry-After")

                # 如果有 "Retry-After" header，則等待指定秒數
                if retry_after:
                    wait_time = int(retry_after)
                    print(f"依照 Retry-After，等待 {wait_time} 秒")
                else:
                    # 如果沒有 "Retry-After" header，使用指數退避 (等待時間會以指數級增長)
                    wait_time = (2 ** retries + random.uniform(0, 1))
                    print(f"使用指數退避：{wait_time:.2f} 秒")

                time.sleep(wait_time)
                retries += 1
                continue

            # 其他 HTTP 錯誤
            response.raise_for_status()

            result = response.json()
            print("成功取得：", result)
            return result

        except requests.exceptions.RequestException as e:
            print("發生錯誤：", e)
            wait_time = (2 ** retries + random.uniform(0, 1))
            print(f"{wait_time:.2f} 秒後重試")
            time.sleep(wait_time)
            retries += 1

    print("達到最大重試次數")

    return None


# 模擬短時間連續請求
print("開始測試")

for request_no in range(1, 8):
    print("\n" + "=" * 30)
    print("client 請求", request_no)
    print("=" * 30)
    fetch_resource_with_retry(max_retries=3)
    time.sleep(0.5)

print("=" * 30)
print("\n測試結束")

開始測試

client 請求 1
嘗試取得資源 (retry=0)
INFO:     127.0.0.1:43378 - "GET /limited-resource HTTP/1.1" 200 OK
成功取得： {'message': 'Success', 'ip': '127.0.0.1', 'current_count': 1, 'remaining_quota': 2}

client 請求 2
嘗試取得資源 (retry=0)
INFO:     127.0.0.1:43384 - "GET /limited-resource HTTP/1.1" 200 OK
成功取得： {'message': 'Success', 'ip': '127.0.0.1', 'current_count': 2, 'remaining_quota': 1}

client 請求 3
嘗試取得資源 (retry=0)
INFO:     127.0.0.1:43400 - "GET /limited-resource HTTP/1.1" 200 OK
成功取得： {'message': 'Success', 'ip': '127.0.0.1', 'current_count': 3, 'remaining_quota': 0}

client 請求 4
嘗試取得資源 (retry=0)
INFO:     127.0.0.1:43410 - "GET /limited-resource HTTP/1.1" 429 Too Many Requests
收到 429 {'detail': 'Too Many Requests', 'ip': '127.0.0.1', 'current_count': 4, 'retry_after_seconds': 9}
依照 Retry-After，等待 9 秒
嘗試取得資源 (retry=1)
INFO:     127.0.0.1:57114 - "GET /limited-resource HTTP/1.1" 200 OK
成功取得： {'message': 'Success', 'ip': '127.0.0.1', 'current_count': 1, 'remaining_quota': 2}

client 請求 5
嘗試取得

## OAuth 授權機制
OAuth（Open Authorization）是一種授權（Authorization）機制，而非「身份驗證（Authentication）」 ，讓使用者可以授權第三方應用程式存取自己的資源，而不需要把帳號密碼直接交給第三方。目前 OAuth 主要版本為 OAuth 2.0。

OAuth 規範中定義了四個主要角色，彼此透過權杖（Token）進行安全的信任交換：
* 資源擁有者 (Resource Owner)：通常指使用者（User），擁有資料並能決定是否授權。
* client (Client)：想要存取使用者資料的第三方應用程式（如網站或手機 App）。
* 授權伺服器 (Authorization Server)：負責驗證使用者身份並核發「存取權杖（Access Token）」的系統（如 Google、Facebook 的驗證中心）。
* 資源伺服器 (Resource Server)：存放使用者實際資料的 API 伺服器，它透過檢查存取權杖來決定是否放行資料。

標準授權碼流程（Authorization Code Flow）
* 引導授權：使用者在某網站點擊「使用 Google 登入」，瀏覽器導向「授權伺服器」。
* 使用者同意：使用者在授權伺服器輸入帳密登入，並勾選同意的存取範圍（Scope，如：讀取個人資料）。
* 發放授權碼：驗證成功後，授權伺服器將一個臨時的授權碼（Authorization Code）傳回給該網站。
* 交換權杖：該網站後端伺服器拿著這個「授權碼」搭配自己的密鑰直接向授權伺服器換取存取權杖（Access Token）。
* 讀取資源：該網站後續只需在 HTTP 請求標頭（Header）中帶上該存取權杖，就能直接向「資源伺服器」請求資料。


### OAuth Server Demo
下面範例為 server 程式，它同時扮演了授權伺服器 (Authorization Server) 和資源伺服器 (Resource Server) 的角色。

在實務上，這兩個角色通常會是獨立的服務，以提供更好的安全性、擴展性和職責分離。但在教學或簡化範例中，將它們合併在一起是很常見的做法。

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel
from typing import Optional
from threading import Thread
import uvicorn
import nest_asyncio

# 應用 nest_asyncio 允許在 Colab 中運行 uvicorn
nest_asyncio.apply()

app = FastAPI()

# 模擬用戶資料庫
class UserInDB(BaseModel):
    username: str
    password: str  # 將 hashed_password 替換為 plain password
    full_name: Optional[str] = None
    email: Optional[str] = None
    mobile: Optional[str] = None


# !!! 安全警告 !!!
# 在實務上，絕不能將密碼以明文形式儲存或處理。
# 本範例僅為簡化演示 OAuth 流程，因此直接使用明文密碼 '111'。
# 請務必在實際應用中使用安全的密碼雜湊方法（例如 bcrypt、scrypt 等）。
users_db = {
    "john": {
        "username": "john",
        "password": "111",
        "full_name": "John Doe",
        "email": "john@google.com",
        "mobile": "0987654321"
    },
    "jane": {
        "username": "jane",
        "password": "222",
        "full_name": "Jane Doe",
        "email": "jane@google.com",
        "mobile": "0912345678"
    },
}


# 從模擬資料庫中獲取用戶的輔助函數
def get_user(username: str):
    if username in users_db:
        user_data = users_db[username]
        return UserInDB(**user_data)
    return None

# 告訴 FastAPI「API 採用 OAuth2 Bearer Token 驗證」，並且取得請求中的 Access Token。
# 「tokenUrl="token"」用來指定核發 Token 的 URL，例如：@app.post("/token")；但只是描述，不會真的去呼叫 /token
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")


# --- OAuth2 身份驗證函數 ---
def verify_password(plain_password: str, stored_password: str) -> bool:
    # 此處僅為展示，實務上不會直接比較明文密碼
    return plain_password == stored_password


# 驗證用戶帳密
def authenticate_user(username: str, password: str):
    user = get_user(username)
    if not user:
        print(f"用戶 '{username}' 未找到。")
        return None
    # 檢查用戶輸入的密碼是否正確
    if not verify_password(password, user.password):
        print(f"用戶 '{username}' 的密碼不正確。")
        return None
    print(f"用戶 '{username}' 身份驗證成功。")
    return user


# 假設權杖有效，回傳一個用戶資料
# 執行 get_current_user() 前，先執行 oauth2_scheme，取得 Access Token，並將結果放入 token 參數
def get_current_user(token: str = Depends(oauth2_scheme)):
    user = get_user(token)
    # 檢查權杖是否有效（在此，token為用戶名）
    if not user:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="無效的身份驗證憑證",
            headers={"WWW-Authenticate": "Bearer"},
        )

    return {"username": user.username,
            "full_name": user.full_name,
            "email": user.email,
            "mobile": user.mobile}


# --- FastAPI 端點 ---
@app.post("/token")
# 此函式扮演授權伺服器
# 自動從 HTTP Request 的 Form Data 中解析 OAuth2 規範要求的輸入欄位，
# 並建立一個 OAuth2PasswordRequestForm 物件後注入給 form_data
async def login_for_access_token(form_data: OAuth2PasswordRequestForm = Depends()):
    user = authenticate_user(form_data.username, form_data.password)
    if not user:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="用戶名或密碼不正確",
            headers={"WWW-Authenticate": "Bearer"},
        )

    # 在此範例中，返回用戶名當作 token。
    access_token = user.username
    return {"access_token": access_token, "token_type": "bearer"}


@app.get("/users/me")
# 此函式扮演資源伺服器
# 在執行 read_users_me() 之前，先執行 get_current_user()，並將回傳值指派給 current_user 參數
async def read_users_me(current_user: dict = Depends(get_current_user)):
    return current_user


# --- 在背景執行緒中運行 FastAPI 伺服器 ---
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8004)

print("正在啟動 OAuth2 FastAPI 伺服器於 http://0.0.0.0:8004")
Thread(target=run_server).start()
print("OAuth2 伺服器已在背景啟動。")
print("要測試，您可以向 /token 發送帶有用戶名和密碼的 POST 請求，然後向 /users/me 發送帶有 Bearer 權杖的 GET 請求。")

正在啟動 OAuth2 FastAPI 伺服器於 http://0.0.0.0:8004
OAuth2 伺服器已在背景啟動。
要測試，您可以向 /token 發送帶有用戶名和密碼的 POST 請求，然後向 /users/me 發送帶有 Bearer 權杖的 GET 請求。


INFO:     Started server process [4647]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


### OAuth Client Demo

In [ ]:
# 匯入 requests 套件，用來發送 HTTP request
import requests

# OAuth2 API server URL
BASE_URL = "http://127.0.0.1:8004" # 與伺服器使用的 port 需一致

# 用戶憑證
USERNAME = input("請輸入用戶名 (例如：john): ")
PASSWORD = input("請輸入密碼 (例如：111): ")

# 獲取存取權杖 (Access Token)
print("\n--- 嘗試獲取存取權杖 ---")
token_url = f"{BASE_URL}/token"

# OAuth2PasswordRequestForm 需要 form-urlencoded 格式的資料
token_data = {
    "username": USERNAME,
    "password": PASSWORD
}

try:
    response = requests.post(token_url, data=token_data)
    response.raise_for_status() # 如果請求失敗，拋出 HTTPError
    token_response = response.json()
    access_token = token_response.get("access_token")
    token_type = token_response.get("token_type")

    if access_token and token_type:
        print(f"成功獲取權杖：{access_token}")
        print(f"權杖類型：{token_type}")
    else:
        print("獲取權杖失敗：", token_response)
        access_token = None # 清除 access_token 以防止後續使用

except requests.exceptions.HTTPError as e:
    print(f"獲取權杖時發生 HTTP 錯誤：{e}")
    print(f"回應內容：{e.response.json()}")
    access_token = None
except requests.exceptions.ConnectionError as e:
    print(f"無法連接到伺服器。請確保伺服器正在運行且地址正確。錯誤：{e}")
    access_token = None
except Exception as e:
    print(f"獲取權杖時發生未知錯誤：{e}")
    access_token = None

# 使用存取權杖存取受保護的資源
if access_token:
    print("\n--- 嘗試使用權杖存取受保護的資源 ---")
    protected_url = f"{BASE_URL}/users/me"
    headers = {
        "Authorization": f"{token_type} {access_token}"
    }

    try:
        response = requests.get(protected_url, headers=headers)
        response.raise_for_status() # 如果請求失敗，拋出 HTTPError
        protected_resource_data = response.json()
        print("成功存取受保護的資源：", protected_resource_data)

    except requests.exceptions.HTTPError as e:
        print(f"存取受保護資源時發生 HTTP 錯誤：{e}")
        print(f"回應內容：{e.response.json()}")
    except requests.exceptions.ConnectionError as e:
        print(f"無法連接到伺服器。請確保伺服器正在運行且地址正確。錯誤：{e}")
    except Exception as e:
        print(f"存取受保護資源時發生未知錯誤：{e}")
else:
    print("無法獲取存取權杖，跳過存取受保護資源。")

請輸入用戶名 (例如：john): john
請輸入密碼 (例如：111): 111

--- 嘗試獲取存取權杖 ---
用戶 'john' 身份驗證成功。
INFO:     127.0.0.1:34656 - "POST /token HTTP/1.1" 200 OK
成功獲取權杖：john
權杖類型：bearer

--- 嘗試使用權杖存取受保護的資源 ---
INFO:     127.0.0.1:34668 - "GET /users/me HTTP/1.1" 200 OK
成功存取受保護的資源： {'username': 'john', 'full_name': 'John Doe', 'email': 'john@google.com', 'mobile': '0987654321'}
